In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
import os
roots=[]
for r,ds,fs in os.walk('/kaggle/input'):
    if 'data_kaggle.yaml' in fs:
        roots.append(r)
print(roots)
IN=roots[0]
open('/tmp/kin.txt','w').write(IN)
new='train: '+IN+'/data/train/images\nval: '+IN+'/data/valid/images\nnc: 7\nnames: [wall, window, door, slide_door, double_door, opening, junction]\n'
open('/tmp/data.yaml','w').write(new)
!ls $IN/data/train/images | wc -l

In [ ]:
import os
IN=open('/tmp/kin.txt').read().strip()
!cp $IN/*.py /kaggle/working/ && ls /kaggle/working/*.py

In [ ]:
# EXP8/H10: focal THAT (1-p)^2 + alpha/class. Retrain small 20ep.
!cd /kaggle/working && python train.py --data /tmp/data.yaml --variant small --imgsz 768 --epochs 20 --batch 8 --lr 2e-4 --out /kaggle/working/runs/vlmdet_small5 2>&1 | tail -25

In [ ]:
# EVAL exp8
!cd /kaggle/working && python eval.py --ckpt /kaggle/working/runs/vlmdet_small5/best.pt --imgdir $(cat /tmp/kin.txt)/data/valid/images --lbldir $(cat /tmp/kin.txt)/data/valid/labels --variant small --conf 0.2 --imgsz 768 2>&1 | tail -12